# 🐍 Pygame: создаём игру «Змейка» — урок 2 из 2
## Еда, рост, очки, столкновения и новая игра

**Возраст:** 10–13 лет  
**Продолжительность:** 80 минут  
**Что уже сделали на уроке 1:** создали окно Pygame, научили змейку двигаться, создали классы `Snake`, `Game` и разложили проект на файлы.

**Сегодня:** не переписываем игру с нуля, а **дополняем тот же проект**. Новый класс — `Food`.

> 🧑‍🏫 **Преподавателю:** текстовые ячейки помогают объяснять материал, а кодовые ячейки `%%writefile` обновляют файлы настоящего проекта. После каждого этапа запускаем игру в отдельном окне. Не переходите к следующему этапу, пока окно не закрыто.


## План занятия (80 минут)

| Время | Этап | Что получится |
|---|---|---|
| 0–10 мин | Повторение урока 1 | Вспомним четыре файла и классы |
| 10–22 мин | Класс `Food` | Яблоко выбирает свободную клетку |
| 22–32 мин | Рисование яблока | Яблоко появляется на поле |
| 32–47 мин | Поедание и рост | Змейка становится длиннее |
| 47–57 мин | Счёт на экране | Видим `Score: 1`, `Score: 2`… |
| 57–67 мин | Столкновение с собой | Игра заканчивается при ошибке |
| 67–77 мин | Перезапуск клавишей R | Полноценный игровой цикл |
| 77–80 мин | Итоги и мини-задание | Обсуждаем код и закрепляем знания |

**Правило:** каждый этап — небольшой шаг. Изучаем только то, что нужно для сегодняшней игры.


---
# 1. Повторяем проект — 0–10 минут

На прошлом занятии мы сделали четыре файла:

```text
snake_oop/
├── main.py       ← запускает игру
├── game.py       ← класс Game: окно и игровой цикл
├── snake.py      ← класс Snake: тело, движение, рисование
└── settings.py   ← размер поля, цвета, скорость
```

Сегодня появится пятый файл `food.py` с классом `Food`.  
**Простой вопрос:** если нужно поменять скорость, какой файл открываем? **Ответ:** `settings.py`.

### Важная особенность Jupyter

Мы редактируем файлы, а затем запускаем **отдельный процесс Python**. Благодаря этому новые версии классов не застревают в памяти Jupyter. Окно игры необходимо закрывать перед следующим запуском.


In [ ]:
from pathlib import Path
import subprocess
import sys

project = Path("snake_oop")
required_files = ["main.py", "game.py", "snake.py", "settings.py"]
missing = [name for name in required_files if not (project / name).exists()]

if missing:
    print("Не хватает файлов урока 1:", ", ".join(missing))
    print("Перенесите сюда папку snake_oop из первого урока или распакуйте архив урока 2.")
else:
    print("✅ Наш проект готов к продолжению!")

def run_game():
    if not (project / "main.py").exists():
        print("Сначала добавьте папку snake_oop с файлами урока 1.")
        return
    # Для преподавателя: запустить свежую версию игры в отдельном процессе.
    subprocess.run([sys.executable, str(project / "main.py")], check=True)


**Проверка:** если появилась строка «✅ Наш проект готов к продолжению!», можно идти дальше.  
Если появилось сообщение об отсутствующих файлах, скопируйте папку `snake_oop` из прошлого урока. В архиве этого урока она уже лежит рядом с ноутбуком.

Для установки библиотеки в используемый Python при необходимости выполните `%pip install pygame` в новой ячейке.

**Что проверяем устно:** `self.body` — список клеток, из которых состоит змейка; `Game.run()` — цикл, который повторяется, пока игра открыта.


---
# 2. Добавляем класс `Food` — 10–22 минуты

Представьте: яблоко — отдельный объект игры. Оно **знает**, где находится (`position`), и **умеет**:

- `relocate(...)` — появляться в новом месте;
- `draw(...)` — рисоваться на экране.

`Food` не знает, как двигать змейку. Это задача другого класса.

### Случайные числа

`random.randint(0, 4)` выбирает одно целое число: 0, 1, 2, 3 или 4. Так мы будем искать случайную клетку поля.


In [ ]:
import random

print("Случайное число:", random.randint(0, 4))
print("Ещё одно:", random.randint(0, 4))


### 2.1. Новые цвета в `settings.py`

Знакомый файл настроек остаётся на месте. Добавим два цвета: для яблока и для текста. **Запустите ячейку — она обновит файл `settings.py`.**


In [ ]:
%%writefile snake_oop/settings.py
# Общие настройки игры
CELL = 20
COLS = 30
ROWS = 20
WIDTH = COLS * CELL
HEIGHT = ROWS * CELL

BACKGROUND = (25, 35, 55)
SNAKE_COLOR = (70, 210, 100)
HEAD_COLOR = (120, 250, 150)
SPEED = 5
FOOD_COLOR = (240, 80, 75)  # Цвет яблока
TEXT_COLOR = (255, 255, 255)  # Цвет текста


### 2.2. Создаём новый файл `food.py`

Обратите внимание на три идеи:

- `self.position` хранит положение яблока в клетках, например `[12, 5]`;
- `if [x, y] not in snake_body` — яблоко не должно появляться внутри змейки;
- `pygame.draw.circle(...)` рисует красный кружок.

> 👩‍🏫 **Для объяснения:** `while True` повторяется, пока не найдём свободную клетку. Команда `break` выходит из цикла. Проверка длины тела защищает от бесконечного поиска на полностью заполненном поле.


In [ ]:
%%writefile snake_oop/food.py
import random
import pygame
from settings import CELL, COLS, ROWS, FOOD_COLOR


class Food:
    def __init__(self, snake_body):
        self.position = None
        self.relocate(snake_body)

    def relocate(self, snake_body):
        # Когда змейка заняла всё поле, поставить яблоко негде.
        if len(snake_body) >= COLS * ROWS:
            self.position = None
            return

        # Ищем случайную свободную клетку.
        while True:
            x = random.randint(0, COLS - 1)
            y = random.randint(0, ROWS - 1)
            if [x, y] not in snake_body:
                self.position = [x, y]
                break

    def draw(self, screen):
        if self.position is None:
            return

        # Превращаем координаты клетки в координаты пикселей.
        x = self.position[0] * CELL + CELL // 2
        y = self.position[1] * CELL + CELL // 2
        pygame.draw.circle(screen, FOOD_COLOR, (x, y), CELL // 2 - 2)


### Маленькая проверка (1 минута)

**Вопросы:** какую информацию хранит `position`? Почему не нужно ставить яблоко в клетку, занятую змейкой?  
**Мини-задание:** найдите строку, которая задаёт цвет яблока; измените цвет и сравните после запуска.


---
# 3. Показываем яблоко на поле — 22–32 минуты

Теперь познакомим `Game` и `Food`:

```python
self.snake = Snake()
self.food = Food(self.snake.body)
```

Далее в игровом цикле вызываем `self.food.draw(self.screen)` рядом с уже знакомым `self.snake.draw(...)`.

**Пока яблоко нельзя съесть.** Сейчас наша цель — только увидеть новый объект в игре.


In [ ]:
%%writefile snake_oop/game.py
import pygame
from settings import WIDTH, HEIGHT, BACKGROUND, SPEED
from snake import Snake
from food import Food  # НОВОЕ: класс яблока находится в food.py


class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Змейка — урок 2: яблоко")
        self.clock = pygame.time.Clock()
        self.snake = Snake()
        self.food = Food(self.snake.body)  # НОВОЕ

    def run(self):
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN:
                    self.snake.turn(event.key)

            self.snake.move()
            if not self.snake.moving:
                pygame.display.set_caption("Змейка у стены! Закройте окно")

            self.screen.fill(BACKGROUND)
            self.food.draw(self.screen)  # НОВОЕ: рисуем яблоко
            self.snake.draw(self.screen)
            pygame.display.flip()
            self.clock.tick(SPEED)

        pygame.quit()


### Проверяем (2 минуты)

Запустите игру. Должны появиться **зелёная змейка и красное круглое яблоко**. Змейкой можно управлять стрелками. При столкновении со стеной движение останавливается, как в первом уроке.

После проверки **закройте окно** и переходите дальше.


In [ ]:
run_game()


---
# 4. Учим змейку есть и расти — 32–47 минут

Вспомним движение из урока 1:

1. Добавляем новую голову в начало списка `body`.
2. Удаляем последний сегмент (хвост).
3. Поэтому длина змейки остаётся такой же.

**Но при съедании яблока хвост удалять НЕ нужно!** Так змейка станет на одну клетку длиннее.

Изменим метод `Snake.move(...)`:

```python
ate = new_head == food_position  # Съели ли яблоко?
self.body.insert(0, new_head)     # Новая голова
if not ate:
    self.body.pop()              # Если не ели, хвост уходит
return ate                       # Сообщаем игре результат
```

`ate` — это `True` или `False`. Класс `Game` по этому ответу понимает, что пора перенести яблоко.


### 4.1. Обновляем `snake.py`

Ищем в новом коде изменившийся метод `move(self, food_position=None)`. Остальные знакомые методы (`turn`, `draw`) сохраняются.


In [ ]:
%%writefile snake_oop/snake.py
import pygame
from settings import CELL, COLS, ROWS, SNAKE_COLOR, HEAD_COLOR


class Snake:
    def __init__(self):
        self.body = [[10, 7], [9, 7], [8, 7]]
        self.direction = "RIGHT"
        self.moving = True

    def turn(self, key):
        if not self.moving:
            return

        if key == pygame.K_UP and self.direction != "DOWN":
            self.direction = "UP"
        elif key == pygame.K_DOWN and self.direction != "UP":
            self.direction = "DOWN"
        elif key == pygame.K_LEFT and self.direction != "RIGHT":
            self.direction = "LEFT"
        elif key == pygame.K_RIGHT and self.direction != "LEFT":
            self.direction = "RIGHT"

    def move(self, food_position=None):
        # Перемещаемся на одну клетку. Возвращаем True, если съели яблоко.
        if not self.moving:
            return False

        new_head = self.body[0].copy()
        if self.direction == "RIGHT":
            new_head[0] += 1
        elif self.direction == "LEFT":
            new_head[0] -= 1
        elif self.direction == "UP":
            new_head[1] -= 1
        elif self.direction == "DOWN":
            new_head[1] += 1

        if not (0 <= new_head[0] < COLS and 0 <= new_head[1] < ROWS):
            self.moving = False
            return False

        ate = new_head == food_position  # Голова вошла в клетку с яблоком?
        self.body.insert(0, new_head)
        if not ate:
            self.body.pop()  # Если не ели, хвост уходит — длина не меняется.
        return ate

    def draw(self, screen):
        for part in self.body:
            x = part[0] * CELL
            y = part[1] * CELL
            pygame.draw.rect(screen, SNAKE_COLOR, (x, y, CELL, CELL))

        head_x = self.body[0][0] * CELL
        head_y = self.body[0][1] * CELL
        pygame.draw.rect(screen, HEAD_COLOR, (head_x, head_y, CELL, CELL))


### 4.2. Обновляем `game.py`

Теперь класс `Game` спрашивает у змейки, съела ли она яблоко. Если да — увеличиваем счётчик и переносим яблоко на свободную клетку. Пока число очков не рисуем: сделаем это следующим шагом.


In [ ]:
%%writefile snake_oop/game.py
import pygame
from settings import WIDTH, HEIGHT, BACKGROUND, SPEED
from snake import Snake
from food import Food


class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Змейка — урок 2: растём!")
        self.clock = pygame.time.Clock()
        self.snake = Snake()
        self.food = Food(self.snake.body)
        self.score = 0  # НОВОЕ: количество съеденных яблок

    def run(self):
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN:
                    self.snake.turn(event.key)

            # Snake.move() сообщает, съели ли мы яблоко.
            ate = self.snake.move(self.food.position)
            if ate:
                self.score += 1
                self.food.relocate(self.snake.body)
                # При полностью заполненном поле это уже победа.
                if self.food.position is None:
                    self.snake.moving = False

            if not self.snake.moving:
                pygame.display.set_caption("Игра остановлена")

            self.screen.fill(BACKGROUND)
            self.food.draw(self.screen)
            self.snake.draw(self.screen)
            pygame.display.flip()
            self.clock.tick(SPEED)

        pygame.quit()


### Проверка (3 минуты)

- Подведите голову змейки к яблоку с помощью стрелок.
- После столкновения с яблоком змейка должна **увеличиться на 1 клетку**.
- Яблоко должно появиться в **новом случайном месте**.

> 🧑‍🏫 Подсказка: если детям трудно доехать до яблока, можно временно поменять `SPEED = 5` на `SPEED = 3` в `settings.py`.


In [ ]:
run_game()


---
# 5. Показываем количество очков — 47–57 минут

Счёт уже хранится в `self.score`, но игрок его не видит. Для текста используем две команды Pygame:

```python
text = self.font.render("Score: 3", True, TEXT_COLOR)
self.screen.blit(text, (10, 8))
```

- `render(...)` превращает строку в картинку с буквами;
- `blit(...)` рисует эту картинку в окне;
- `(10, 8)` — место текста в пикселях, от левого верхнего угла.

**Важно:** для простоты игры надпись показывается прямо поверх верхней части поля. Движению она не мешает.


In [ ]:
%%writefile snake_oop/game.py
import pygame
from settings import WIDTH, HEIGHT, BACKGROUND, SPEED, TEXT_COLOR
from snake import Snake
from food import Food


class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Змейка — урок 2: очки")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.Font(None, 32)  # НОВОЕ: шрифт
        self.snake = Snake()
        self.food = Food(self.snake.body)
        self.score = 0

    def run(self):
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN:
                    self.snake.turn(event.key)

            ate = self.snake.move(self.food.position)
            if ate:
                self.score += 1
                self.food.relocate(self.snake.body)
                if self.food.position is None:
                    self.snake.moving = False

            self.screen.fill(BACKGROUND)
            self.food.draw(self.screen)
            self.snake.draw(self.screen)

            # НОВОЕ: пишем число очков в верхней части поля.
            text = self.font.render(f"Score: {self.score}", True, TEXT_COLOR)
            self.screen.blit(text, (10, 8))
            pygame.display.flip()
            self.clock.tick(SPEED)

        pygame.quit()


### Проверка (2 минуты)

Соберите хотя бы два яблока. В левом верхнем углу надпись должна измениться с `Score: 0` до `Score: 2`. Посмотрите, как значение меняется после каждого яблока.

**Мини-задание:** поменяйте текст `Score` на `Apples` и посмотрите, что изменилось. (Английский текст выбран, чтобы он гарантированно отображался стандартным шрифтом Pygame.)


In [ ]:
run_game()


---
# 6. Змейка не должна врезаться в себя — 57–67 минут

Чем длиннее змейка, тем сложнее управлять ей. Добавим проверку столкновения с собственным телом **перед шагом**.

```python
body_to_check = self.body if ate else self.body[:-1]
if new_head in body_to_check:
    self.moving = False
    return False
```

**Что означает `self.body[:-1]`?** Все сегменты, кроме последнего (хвоста). Если мы не едим яблоко, хвост на этом шаге уйдёт. Поэтому его прежняя клетка свободна. Если едим яблоко, хвост остаётся, и проверять нужно все сегменты.

Это один из самых интересных моментов игры: мы не просто рисуем квадрат, а проверяем **правила игрового мира**.


In [ ]:
%%writefile snake_oop/snake.py
import pygame
from settings import CELL, COLS, ROWS, SNAKE_COLOR, HEAD_COLOR


class Snake:
    def __init__(self):
        self.body = [[10, 7], [9, 7], [8, 7]]
        self.direction = "RIGHT"
        self.moving = True

    def turn(self, key):
        if not self.moving:
            return

        if key == pygame.K_UP and self.direction != "DOWN":
            self.direction = "UP"
        elif key == pygame.K_DOWN and self.direction != "UP":
            self.direction = "DOWN"
        elif key == pygame.K_LEFT and self.direction != "RIGHT":
            self.direction = "LEFT"
        elif key == pygame.K_RIGHT and self.direction != "LEFT":
            self.direction = "RIGHT"

    def move(self, food_position=None):
        if not self.moving:
            return False

        new_head = self.body[0].copy()
        if self.direction == "RIGHT":
            new_head[0] += 1
        elif self.direction == "LEFT":
            new_head[0] -= 1
        elif self.direction == "UP":
            new_head[1] -= 1
        elif self.direction == "DOWN":
            new_head[1] += 1

        if not (0 <= new_head[0] < COLS and 0 <= new_head[1] < ROWS):
            self.moving = False
            return False

        ate = new_head == food_position
        # НОВОЕ: проверяем, не заняла ли голова клетку тела.
        # Если яблоко не съедено, хвост уйдёт, и его клетка разрешена.
        body_to_check = self.body if ate else self.body[:-1]
        if new_head in body_to_check:
            self.moving = False
            return False

        self.body.insert(0, new_head)
        if not ate:
            self.body.pop()
        return ate

    def draw(self, screen):
        for part in self.body:
            x = part[0] * CELL
            y = part[1] * CELL
            pygame.draw.rect(screen, SNAKE_COLOR, (x, y, CELL, CELL))

        head_x = self.body[0][0] * CELL
        head_y = self.body[0][1] * CELL
        pygame.draw.rect(screen, HEAD_COLOR, (head_x, head_y, CELL, CELL))


### Пробуем столкновение (2 минуты)

Соберите несколько яблок, а затем попытайтесь провести голову через тело. После столкновения змейка должна остановиться. Со стеной работает прежнее правило — движение тоже прекращается.

Если хочется быстро проверить правило, преподаватель может временно сделать начальное тело длиннее в `Snake.__init__()`, а после проверки вернуть исходные три сегмента.


In [ ]:
run_game()


---
# 7. Экран Game Over и перезапуск — 67–77 минут

Сейчас после ошибки змейка просто останавливается. Сделаем более понятное завершение игры:

- надпись **Game over!**;
- подсказка **Press R to restart**;
- клавиша **R** создаёт новую змейку, яблоко и сбрасывает очки;
- если змейка займёт все клетки поля, показываем **You win!**.

### Новые методы класса `Game`

- `reset()` — начать игру заново;
- `draw()` — нарисовать всё, что должно быть на экране;
- `run()` — как раньше, выполняет игровой цикл.

Посмотрите, как стало удобнее читать класс: каждый метод отвечает за свою задачу.


In [ ]:
%%writefile snake_oop/game.py
import pygame
from settings import WIDTH, HEIGHT, BACKGROUND, SPEED, TEXT_COLOR
from snake import Snake
from food import Food


class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Змейка — готовая игра")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.Font(None, 32)
        self.big_font = pygame.font.Font(None, 44)
        self.reset()  # НОВОЕ: создаём новую игру

    def reset(self):
        # НОВОЕ: с самого начала создаём змейку, яблоко и нулевой счёт.
        self.snake = Snake()
        self.food = Food(self.snake.body)
        self.score = 0
        self.won = False

    def draw(self):
        self.screen.fill(BACKGROUND)
        self.food.draw(self.screen)
        self.snake.draw(self.screen)

        score_text = self.font.render(f"Score: {self.score}", True, TEXT_COLOR)
        self.screen.blit(score_text, (10, 8))

        # НОВОЕ: после поражения или победы показываем сообщение.
        if not self.snake.moving:
            box = pygame.Rect(WIDTH // 2 - 175, HEIGHT // 2 - 50, 350, 100)
            pygame.draw.rect(self.screen, (12, 18, 30), box)
            title = "You win!" if self.won else "Game over!"
            msg = self.big_font.render(title, True, TEXT_COLOR)
            hint = self.font.render("Press R to restart", True, TEXT_COLOR)
            self.screen.blit(msg, msg.get_rect(center=(WIDTH // 2, HEIGHT // 2 - 18)))
            self.screen.blit(hint, hint.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 24)))

        pygame.display.flip()

    def run(self):
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_r and not self.snake.moving:
                        self.reset()  # НОВОЕ: начинаем заново
                    else:
                        self.snake.turn(event.key)

            if self.snake.moving:
                ate = self.snake.move(self.food.position)
                if ate:
                    self.score += 1
                    self.food.relocate(self.snake.body)
                    if self.food.position is None:
                        self.won = True
                        self.snake.moving = False

            self.draw()
            self.clock.tick(SPEED)

        pygame.quit()


### Финальный запуск (3 минуты)

Управление: **стрелки** — движение. Если врезались — **R** для нового раунда. Окно закрываем крестиком.

**Проверьте по очереди:** 1) яблоко видно; 2) змейка растёт; 3) очки растут; 4) стены останавливают; 5) тело останавливает; 6) R перезапускает с нуля.


In [ ]:
run_game()


---
# 8. Мини-задания и итоги — 77–80 минут

Если осталось время, выберите **одно** задание:

1. Измените цвет яблока в `settings.py`.
2. Уменьшите скорость `SPEED`, чтобы играть было проще.
3. Измените сообщение `Game over!` на `Try again!`.
4. Попробуйте сделать яблоко немного меньше: найдите `CELL // 2 - 2` в `food.py`.

### Что мы теперь знаем

| Класс | За что отвечает |
|---|---|
| `Snake` | Где тело, куда двигаться, как расти и когда остановиться |
| `Food` | Где находится яблоко, куда перенести, как нарисовать |
| `Game` | Запуск окна, события клавиш, подсчёт очков и перезапуск |

**Итог урока:** игра готова! Теперь мы умеем дополнять уже существующий проект, не смешивая весь код в одном файле.


---
# Подсказки преподавателю и разбор ошибок

**Ошибка `No module named pygame`** — запустите `%pip install pygame`, затем перезапустите ядро Jupyter.  
**Ошибка `No module named food`** — убедитесь, что `food.py` создан в `snake_oop` (рядом с `game.py`).  
**Окно не появляется** — нужен **локальный** Jupyter/VS Code с графическим рабочим столом. Google Colab не подходит для обычных окон Pygame.  
**Изменения файла не видны** — закройте игровое окно и вызовите `run_game()` снова: он запускает новый процесс Python.  
**Змейка слишком быстрая** — поставьте `SPEED = 3`.  
**Змейка стоит после удара** — это правило игры; теперь можно нажать `R` для перезапуска.  
**Яблоко случайно оказалось «под» змейкой?** — `Food.relocate()` проверяет занятые клетки, поэтому новый объект не должен там появляться.

### Небольшие вопросы для устной проверки

- Чем метод `Food.draw()` отличается от `Snake.move()`?
- Зачем `Snake.move()` возвращает `True` или `False`?
- Когда хвост змейки **не** нужно удалять?
- Зачем нужен `Game.reset()`?
- В каком файле поменять скорость или цвет?

### Для желающих: самостоятельная работа после урока

Сделайте третий цвет для головы, измените размер игрового поля, настройте скорость так, чтобы новичку было удобно играть. Не добавляйте новые сложные классы, пока все три существующих не стали понятными.
